# 4. Provider Performance & Network Analytics
## Vertically Complex Provider Intelligence Pipeline

Strategy: LIMIT 1000 → Local CSV → Pandas → Spark

## Available Tables:
- **OMOP**: 24 tables (provider, care_site, procedure_occurrence, drug_exposure, condition_occurrence, observation, person, death)
- **Medicare**: 6 tables (physicians_and_other_supplier)
- **NPPES**: 1 table (healthcare_provider_taxonomy_code_set)

## Pipeline Architecture:
- **Bronze**: 10 tables (raw data)
- **Silver**: 6 intermediate layers (provider profiling)
- **Gold**: 15 vertical layers → 4 final metrics

## Final Metrics:
1. **Provider Quality Composite** - Quality indicator integration
2. **Network Efficiency Score** - Referral network efficiency
3. **Patient Outcome Attribution** - Provider-specific outcome contribution
4. **Care Coordination Index** - Collaborative care quality

In [124]:
!pip install google-cloud-bigquery
!pip install pandas
!pip install networkx


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [125]:
from google.cloud import bigquery
import pandas as pd
from pyspark.sql import SparkSession, functions as F, Window as W
from pyspark.sql.types import *
import os
from datetime import datetime
import networkx as nx
import json
import requests 

In [126]:
print("Initializing Spark...")
spark = (
    SparkSession.builder 
    .appName("CMS_ProviderPerformance") 
    .master("local[*]") 

    # OpenLineage Configuration
    .config("spark.jars.packages", "io.openlineage:openlineage-spark_2.12:1.18.0")
    .config("spark.extraListeners", "io.openlineage.spark.agent.OpenLineageSparkListener")
    .config("spark.openlineage.transport.type", "http")
    .config("spark.openlineage.transport.url", "http://localhost:4601")
    .config("spark.openlineage.namespace", "provider_performance")

    .config("spark.driver.memory", "4g") 
    .config("spark.executor.memory", "4g") 
    .config("spark.sql.shuffle.partitions", "8") 
    .config("spark.driver.maxResultSize", "2g") 
    .getOrCreate()
)

print(f"Spark version: {spark.version}")
print(f"Spark UI: {spark.sparkContext.uiWebUrl}")
print(f"✅ OpenLineage → http://localhost:4601")
print(f"✅ Namespace: provider_performance")

Initializing Spark...
Spark version: 3.5.1
Spark UI: http://mac:4041
✅ OpenLineage → http://localhost:4601
✅ Namespace: provider_performance


In [127]:
# Check Marquez Connection
print("\nChecking Marquez connection...")
try:
    response = requests.get("http://localhost:4601/api/v1/namespaces", timeout=2)
    if response.status_code == 200:
        print("✅ Marquez is running at http://localhost:4601")
        print("✅ Web UI: http://localhost:3601")
    else:
        print("⚠️ Marquez responded but might have issues")
except Exception as e:
    print("❌ Cannot connect to Marquez!")
    print("   Please run: docker-compose -f docker-compose-enhanced.yml up -d")


Checking Marquez connection...
✅ Marquez is running at http://localhost:4601
✅ Web UI: http://localhost:3601


In [128]:
PROJECT_ID = "opportune-ruler-447319-b3"
LIMIT = 1000
LOCAL_DATA_DIR = "./4_data"

# os.makedirs(LOCAL_DATA_DIR, exist_ok=True)

DATASETS = {
    "omop": "bigquery-public-data.cms_synthetic_patient_data_omop",
    "medicare": "bigquery-public-data.cms_medicare",
    "nppes": "bigquery-public-data.nppes"
}

print(f"Config:")
print(f"  Project: {PROJECT_ID}")
print(f"  Limit: {LIMIT} rows per table")
print(f"  Data dir: {LOCAL_DATA_DIR}")

Config:
  Project: opportune-ruler-447319-b3
  Limit: 1000 rows per table
  Data dir: ./4_data


# STEP 1: Download from BigQuery (LIMIT 1000)

In [129]:
def download_table(client, dataset_key, table_name, limit=1000):
    """Download table from BigQuery to local CSV"""
    try:
        source = DATASETS[dataset_key]
        query = f"SELECT * FROM `{source}.{table_name}` LIMIT {limit}"
        df = client.query(query).to_dataframe()
        
        output_file = f"{LOCAL_DATA_DIR}/{dataset_key}_{table_name}.csv"
        df.to_csv(output_file, index=False)
        print(f"  ✓ {dataset_key}.{table_name}: {len(df)} rows → {output_file}")
        return True
    except Exception as e:
        print(f"  ✗ {dataset_key}.{table_name}: {str(e)}")
        return False

In [130]:
print("\n" + "="*60)
print("DOWNLOADING TABLES")
print("="*60)

client = bigquery.Client(project=PROJECT_ID)

tables_to_download = [
    ("omop", "provider"),
    ("omop", "care_site"),
    ("omop", "person"),
    ("omop", "death"),
    ("omop", "procedure_occurrence"),
    ("omop", "drug_exposure"),
    ("omop", "condition_occurrence"),
    ("omop", "observation"),
    ("medicare", "physicians_and_other_supplier_2014"),
    ("nppes", "healthcare_provider_taxonomy_code_set")
]

for dataset_key, table_name in tables_to_download:
    download_table(client, dataset_key, table_name, LIMIT)

print(f"\n✓ Downloaded {len(tables_to_download)} tables")


DOWNLOADING TABLES
  ✗ omop.provider: 403 POST https://bigquery.googleapis.com/bigquery/v2/projects/opportune-ruler-447319-b3/jobs?prettyPrint=false: Access Denied: Project opportune-ruler-447319-b3: User does not have bigquery.jobs.create permission in project opportune-ruler-447319-b3.

Location: None
Job ID: 429ab5b9-4a79-4420-a62e-1a6ec265d305

  ✗ omop.care_site: 403 POST https://bigquery.googleapis.com/bigquery/v2/projects/opportune-ruler-447319-b3/jobs?prettyPrint=false: Access Denied: Project opportune-ruler-447319-b3: User does not have bigquery.jobs.create permission in project opportune-ruler-447319-b3.

Location: None
Job ID: 3c85e6ba-7b84-4ece-8815-97d9957a6f01

  ✗ omop.person: 403 POST https://bigquery.googleapis.com/bigquery/v2/projects/opportune-ruler-447319-b3/jobs?prettyPrint=false: Access Denied: Project opportune-ruler-447319-b3: User does not have bigquery.jobs.create permission in project opportune-ruler-447319-b3.

Location: None
Job ID: f92bb137-7261-4a67-a43f

# STEP 2: Load CSV → Pandas → Spark (Bronze Layer)

In [131]:
def load_csv_to_spark_with_lineage(dataset_key, table_name, layer="bronze"):
    """Load CSV file into Spark - NO metadata columns, NO count()"""
    try:
        csv_path = f"{LOCAL_DATA_DIR}/{dataset_key}_{table_name}.csv"
        
        if not os.path.exists(csv_path):
            print(f"  ✗ {dataset_key}.{table_name}: CSV not found at {csv_path}")
            return None
        
        # Read CSV - NO metadata, NO count()
        df = spark.read \
            .option("header", "true") \
            .option("inferSchema", "true") \
            .csv(csv_path)
        
        print(f"  ✓ {dataset_key}.{table_name}: loaded into {layer} layer")
        return df
        
    except Exception as e:
        print(f"  ✗ {dataset_key}.{table_name}: Error - {str(e)}")
        return None


def add_layer_metadata(df, layer, source_tables, target_table=None):
    """DEPRECATED - Don't use metadata columns"""
    return df  # Just return DataFrame as-is

In [132]:
print("\n" + "="*60)
print("LOADING BRONZE LAYER")
print("="*60)

os.makedirs("./output/bronze", exist_ok=True)

def load_and_write_bronze(dataset_key, table_name):
    """Load CSV and write immediately to capture lineage"""
    df = load_csv_to_spark_with_lineage(dataset_key, table_name, "bronze")
    if df is not None:
        output_path = f"./output/bronze/bronze_{dataset_key}_{table_name}"
        df.write.mode("overwrite").parquet(output_path)
        print(f"    → Written to bronze")
        # Read back for use in Silver
        return spark.read.parquet(output_path)
    return None

bronze_provider = load_and_write_bronze("omop", "provider")
bronze_care_site = load_and_write_bronze("omop", "care_site")
bronze_person = load_and_write_bronze("omop", "person")
bronze_death = load_and_write_bronze("omop", "death")
bronze_procedure = load_and_write_bronze("omop", "procedure_occurrence")
bronze_drug = load_and_write_bronze("omop", "drug_exposure")
bronze_condition = load_and_write_bronze("omop", "condition_occurrence")
bronze_observation = load_and_write_bronze("omop", "observation")
bronze_medicare = load_and_write_bronze("medicare", "physicians_and_other_supplier_2014")
bronze_taxonomy = load_and_write_bronze("nppes", "healthcare_provider_taxonomy_code_set")

print("\n✓ Bronze layer loaded (10 tables)")


LOADING BRONZE LAYER
  ✓ omop.provider: loaded into bronze layer
    → Written to bronze
  ✓ omop.care_site: loaded into bronze layer
    → Written to bronze


25/11/23 22:24:36 ERROR ContextFactory: Query execution is null: can't emit event for executionId 321
25/11/23 22:24:36 ERROR ContextFactory: Query execution is null: can't emit event for executionId 321
25/11/23 22:24:36 ERROR ContextFactory: Query execution is null: can't emit event for executionId 322
25/11/23 22:24:36 ERROR ContextFactory: Query execution is null: can't emit event for executionId 322
25/11/23 22:24:36 ERROR ContextFactory: Query execution is null: can't emit event for executionId 323
25/11/23 22:24:36 ERROR ContextFactory: Query execution is null: can't emit event for executionId 323
25/11/23 22:24:36 ERROR ContextFactory: Query execution is null: can't emit event for executionId 324
25/11/23 22:24:36 ERROR ContextFactory: Query execution is null: can't emit event for executionId 324


  ✓ omop.person: loaded into bronze layer
    → Written to bronze


25/11/23 22:24:36 ERROR ContextFactory: Query execution is null: can't emit event for executionId 325
25/11/23 22:24:36 ERROR ContextFactory: Query execution is null: can't emit event for executionId 325


  ✓ omop.death: loaded into bronze layer
    → Written to bronze
  ✓ omop.procedure_occurrence: loaded into bronze layer


...

    → Written to bronze
  ✓ omop.drug_exposure: loaded into bronze layer
    → Written to bronze


25/11/23 22:24:36 ERROR ContextFactory: Query execution is null: can't emit event for executionId 329
25/11/23 22:24:36 ERROR ContextFactory: Query execution is null: can't emit event for executionId 329
25/11/23 22:24:36 ERROR ContextFactory: Query execution is null: can't emit event for executionId 330
25/11/23 22:24:36 ERROR ContextFactory: Query execution is null: can't emit event for executionId 330
25/11/23 22:24:36 ERROR ContextFactory: Query execution is null: can't emit event for executionId 331
25/11/23 22:24:36 ERROR ContextFactory: Query execution is null: can't emit event for executionId 331


  ✓ omop.condition_occurrence: loaded into bronze layer
    → Written to bronze
  ✓ omop.observation: loaded into bronze layer
    → Written to bronze
  ✓ medicare.physicians_and_other_supplier_2014: loaded into bronze layer
    → Written to bronze


25/11/23 22:24:36 ERROR ContextFactory: Query execution is null: can't emit event for executionId 333
25/11/23 22:24:36 ERROR ContextFactory: Query execution is null: can't emit event for executionId 333
25/11/23 22:24:37 ERROR ContextFactory: Query execution is null: can't emit event for executionId 334
25/11/23 22:24:37 ERROR ContextFactory: Query execution is null: can't emit event for executionId 334
25/11/23 22:24:37 ERROR ContextFactory: Query execution is null: can't emit event for executionId 335
25/11/23 22:24:37 ERROR ContextFactory: Query execution is null: can't emit event for executionId 335


  ✓ nppes.healthcare_provider_taxonomy_code_set: loaded into bronze layer
    → Written to bronze

✓ Bronze layer loaded (10 tables)


# STEP 3: Build Silver Layer (6 Intermediate Tables)

In [ ]:
print("\n" + "="*60)
print("SILVER 1: Provider Demographics Enriched")
print("="*60)

os.makedirs("./output/silver", exist_ok=True)

silver_provider_demographics = bronze_provider \
    .join(bronze_care_site, bronze_provider.care_site_id == bronze_care_site.care_site_id, "left") \
    .select(
        bronze_provider.provider_id,
        bronze_provider.provider_name,
        bronze_provider.npi,
        bronze_provider.dea,
        bronze_provider.specialty_concept_id,
        bronze_provider.specialty_source_value,
        bronze_provider.care_site_id,
        bronze_provider.year_of_birth,
        bronze_provider.gender_concept_id,
        bronze_care_site.care_site_name,
        bronze_care_site.place_of_service_concept_id,
        F.lit(datetime.now().isoformat()).alias("processed_timestamp")
    )

print(f"✓ Provider demographics: {silver_provider_demographics.count()} providers")
silver_provider_demographics.show(5, truncate=False)

silver_provider_demographics = add_layer_metadata(
    df=silver_provider_demographics,
    layer="silver",
    source_tables=["bronze_provider", "bronze_care_site"],
    target_table="silver_provider_demographics"
)

# Write immediately!
silver_provider_demographics.write.mode("overwrite").parquet("./output/silver/silver_provider_demographics")
print(f"✓ PD written")

# Read back
silver_provider_demographics = spark.read.parquet("./output/silver/silver_provider_demographics")
print(f"  PD: {silver_provider_demographics.count()}")


SILVER 1: Provider Demographics Enriched
✓ Provider demographics: 1000 providers
+-----------+-------------+----------+----+--------------------+----------------------+------------+-------------+-----------------+--------------+---------------------------+--------------------------+
|provider_id|provider_name|npi       |dea |specialty_concept_id|specialty_source_value|care_site_id|year_of_birth|gender_concept_id|care_site_name|place_of_service_concept_id|processed_timestamp       |
+-----------+-------------+----------+----+--------------------+----------------------+------------+-------------+-----------------+--------------+---------------------------+--------------------------+
|135687     |NULL         |9232459980|NULL|NULL                |NULL                  |65536       |NULL         |NULL             |NULL          |NULL                       |2025-11-23T22:24:37.325438|
|135691     |NULL         |6537458182|NULL|NULL                |NULL                  |65536       |NULL  

25/11/23 22:24:37 ERROR ContextFactory: Query execution is null: can't emit event for executionId 337
25/11/23 22:24:37 ERROR ContextFactory: Query execution is null: can't emit event for executionId 337
25/11/23 22:24:37 ERROR ContextFactory: Query execution is null: can't emit event for executionId 338
25/11/23 22:24:37 ERROR ContextFactory: Query execution is null: can't emit event for executionId 338
25/11/23 22:24:37 ERROR ContextFactory: Query execution is null: can't emit event for executionId 339
25/11/23 22:24:37 ERROR ContextFactory: Query execution is null: can't emit event for executionId 339
25/11/23 22:24:37 ERROR ContextFactory: Query execution is null: can't emit event for executionId 339
25/11/23 22:24:37 ERROR ContextFactory: Query execution is null: can't emit event for executionId 339


  PD: 1000


25/11/23 22:24:37 ERROR ContextFactory: Query execution is null: can't emit event for executionId 341
25/11/23 22:24:37 ERROR ContextFactory: Query execution is null: can't emit event for executionId 341
25/11/23 22:24:37 ERROR ContextFactory: Query execution is null: can't emit event for executionId 341


In [134]:
print("\n" + "="*60)
print("SILVER 2: Provider Clinical Activity")
print("="*60)

procedure_activity = bronze_procedure \
    .groupBy("provider_id") \
    .agg(
        F.count("*").alias("procedure_count"),
        F.countDistinct("person_id").alias("unique_patients_procedures"),
        F.countDistinct("procedure_concept_id").alias("procedure_variety")
    )

drug_activity = bronze_drug \
    .groupBy("provider_id") \
    .agg(
        F.count("*").alias("drug_count"),
        F.countDistinct("person_id").alias("unique_patients_drugs"),
        F.countDistinct("drug_concept_id").alias("drug_variety")
    )

condition_activity = bronze_condition \
    .groupBy("provider_id") \
    .agg(
        F.count("*").alias("condition_count"),
        F.countDistinct("person_id").alias("unique_patients_conditions"),
        F.countDistinct("condition_concept_id").alias("condition_variety")
    )

silver_provider_activity = procedure_activity \
    .join(drug_activity, "provider_id", "full_outer") \
    .join(condition_activity, "provider_id", "full_outer") \
    .fillna(0) \
    .withColumn("total_encounters", 
                F.col("procedure_count") + F.col("drug_count") + F.col("condition_count")) \
    .withColumn("total_unique_patients",
                F.greatest(F.col("unique_patients_procedures"), 
                          F.col("unique_patients_drugs"),
                          F.col("unique_patients_conditions"))) \
    .withColumn("processed_timestamp", F.lit(datetime.now().isoformat()))

print(f"✓ Provider activity: {silver_provider_activity.count()} providers")
silver_provider_activity.show(5, truncate=False)

silver_provider_activity = add_layer_metadata(
    df=silver_provider_activity,
    layer="silver",
    source_tables=["bronze_procedure", "bronze_drug", "bronze_condition"],
    target_table="silver_provider_activity"
)

# Write immediately!
silver_provider_activity.write.mode("overwrite").parquet("./output/silver/silver_provider_activity")
print(f"✓ PA written")

# Read back
silver_provider_activity = spark.read.parquet("./output/silver/silver_provider_activity")
print(f"  PA: {silver_provider_activity.count()}")


SILVER 2: Provider Clinical Activity


25/11/23 22:24:37 ERROR ContextFactory: Query execution is null: can't emit event for executionId 342
25/11/23 22:24:37 ERROR ContextFactory: Query execution is null: can't emit event for executionId 342
25/11/23 22:24:37 ERROR ContextFactory: Query execution is null: can't emit event for executionId 342


✓ Provider activity: 2408 providers
+-----------+---------------+--------------------------+-----------------+----------+---------------------+------------+---------------+--------------------------+-----------------+----------------+---------------------+--------------------------+
|provider_id|procedure_count|unique_patients_procedures|procedure_variety|drug_count|unique_patients_drugs|drug_variety|condition_count|unique_patients_conditions|condition_variety|total_encounters|total_unique_patients|processed_timestamp       |
+-----------+---------------+--------------------------+-----------------+----------+---------------------+------------+---------------+--------------------------+-----------------+----------------+---------------------+--------------------------+
|0          |12             |12                        |5                |0         |0                    |0           |0              |0                         |0                |12              |12                   |

In [135]:
print("\n" + "="*60)
print("SILVER 3: Patient Outcomes by Provider")
print("="*60)

patient_outcomes = bronze_person \
    .join(bronze_death, "person_id", "left") \
    .join(bronze_condition, "person_id", "left") \
    .groupBy(bronze_condition.provider_id) \
    .agg(
        F.count(bronze_person.person_id).alias("total_patients"),
        F.sum(F.when(bronze_death.person_id.isNotNull(), 1).otherwise(0)).alias("deceased_patients"),
        F.avg(F.datediff(bronze_condition.condition_end_date, bronze_condition.condition_start_date)).alias("avg_condition_duration_days"),
        F.countDistinct(bronze_condition.condition_concept_id).alias("condition_complexity")
    ) \
    .withColumn("mortality_rate", 
                F.when(F.col("total_patients") > 0,
                      F.col("deceased_patients") / F.col("total_patients")).otherwise(0)) \
    .withColumn("processed_timestamp", F.lit(datetime.now().isoformat()))

silver_patient_outcomes = patient_outcomes

print(f"✓ Patient outcomes: {silver_patient_outcomes.count()} providers")
silver_patient_outcomes.show(5, truncate=False)

silver_patient_outcomes = add_layer_metadata(
    df=silver_patient_outcomes,
    layer="silver",
    source_tables=["bronze_person", "bronze_death", "bronze_condition"],
    target_table="silver_patient_outcomes"
)

# Write immediately!
silver_patient_outcomes.write.mode("overwrite").parquet("./output/silver/silver_patient_outcomes")
print(f"✓ PO written")

# Read back
silver_patient_outcomes = spark.read.parquet("./output/silver/silver_patient_outcomes")
print(f"  PO: {silver_patient_outcomes.count()}")


SILVER 3: Patient Outcomes by Provider
✓ Patient outcomes: 2 providers
+-----------+--------------+-----------------+---------------------------+--------------------+--------------------+--------------------------+
|provider_id|total_patients|deceased_patients|avg_condition_duration_days|condition_complexity|mortality_rate      |processed_timestamp       |
+-----------+--------------+-----------------+---------------------------+--------------------+--------------------+--------------------------+
|9069       |1             |0                |3.0                        |1                   |0.0                 |2025-11-23T22:24:38.373505|
|NULL       |999           |1                |NULL                       |0                   |0.001001001001001001|2025-11-23T22:24:38.373505|
+-----------+--------------+-----------------+---------------------------+--------------------+--------------------+--------------------------+

✓ PO written
  PO: 2


25/11/23 22:24:38 ERROR ContextFactory: Query execution is null: can't emit event for executionId 345
25/11/23 22:24:38 ERROR ContextFactory: Query execution is null: can't emit event for executionId 345
25/11/23 22:24:38 ERROR ContextFactory: Query execution is null: can't emit event for executionId 345
25/11/23 22:24:38 ERROR ContextFactory: Query execution is null: can't emit event for executionId 345
25/11/23 22:24:38 ERROR ContextFactory: Query execution is null: can't emit event for executionId 345
25/11/23 22:24:38 ERROR ContextFactory: Query execution is null: can't emit event for executionId 345
25/11/23 22:24:38 ERROR ContextFactory: Query execution is null: can't emit event for executionId 345
25/11/23 22:24:38 ERROR ContextFactory: Query execution is null: can't emit event for executionId 345
25/11/23 22:24:38 ERROR ContextFactory: Query execution is null: can't emit event for executionId 345
25/11/23 22:24:38 ERROR ContextFactory: Query execution is null: can't emit event 

In [136]:
print("\n" + "="*60)
print("SILVER 4: Provider Referral Network")
print("="*60)

procedure_referrals = bronze_procedure \
    .select("person_id", "provider_id", "procedure_dat") \
    .withColumnRenamed("provider_id", "provider_from")

drug_providers = bronze_drug \
    .select("person_id", "provider_id", "drug_exposure_start_date") \
    .withColumnRenamed("provider_id", "provider_to") \
    .withColumnRenamed("drug_exposure_start_date", "referral_date")

referral_network = procedure_referrals \
    .join(drug_providers, "person_id") \
    .filter(F.col("provider_from") != F.col("provider_to")) \
    .groupBy("provider_from", "provider_to") \
    .agg(
        F.count("*").alias("referral_count"),
        F.countDistinct("person_id").alias("unique_patients_referred")
    )

outbound_referrals = referral_network \
    .groupBy("provider_from") \
    .agg(
        F.count("*").alias("outbound_referral_count"),
        F.sum("referral_count").alias("total_patients_referred_out")
    )

inbound_referrals = referral_network \
    .groupBy("provider_to") \
    .agg(
        F.count("*").alias("inbound_referral_count"),
        F.sum("referral_count").alias("total_patients_referred_in")
    )

silver_referral_network = outbound_referrals \
    .withColumnRenamed("provider_from", "provider_id") \
    .join(inbound_referrals.withColumnRenamed("provider_to", "provider_id"), "provider_id", "full_outer") \
    .fillna(0) \
    .withColumn("network_centrality", 
                F.col("outbound_referral_count") + F.col("inbound_referral_count")) \
    .withColumn("processed_timestamp", F.lit(datetime.now().isoformat()))

print(f"✓ Referral network: {silver_referral_network.count()} providers")
silver_referral_network.show(5, truncate=False)

silver_referral_network = add_layer_metadata(
    df=silver_referral_network,
    layer="silver",
    source_tables=["bronze_procedure", "bronze_drug"],
    target_table="silver_referral_network"
)

# Write immediately!
silver_referral_network.write.mode("overwrite").parquet("./output/silver/silver_referral_network")
print(f"✓ RN written")

# Read back
silver_referral_network = spark.read.parquet("./output/silver/silver_referral_network")
print(f"  RN: {silver_referral_network.count()}")

25/11/23 22:24:38 ERROR ContextFactory: Query execution is null: can't emit event for executionId 350
25/11/23 22:24:38 ERROR ContextFactory: Query execution is null: can't emit event for executionId 350
25/11/23 22:24:38 ERROR ContextFactory: Query execution is null: can't emit event for executionId 350



SILVER 4: Provider Referral Network
✓ Referral network: 6 providers
+-----------+-----------------------+---------------------------+----------------------+--------------------------+------------------+--------------------------+
|provider_id|outbound_referral_count|total_patients_referred_out|inbound_referral_count|total_patients_referred_in|network_centrality|processed_timestamp       |
+-----------+-----------------------+---------------------------+----------------------+--------------------------+------------------+--------------------------+
|2317       |0                      |0                          |1                     |1                         |1                 |2025-11-23T22:24:38.793105|
|2632       |1                      |1                          |0                     |0                         |1                 |2025-11-23T22:24:38.793105|
|11087      |0                      |0                          |1                     |1                         |1     

In [137]:
print("\n" + "="*60)
print("SILVER 5: Provider Quality Indicators")
print("="*60)

readmission_proxy = bronze_condition \
    .withColumn("next_visit", F.lead("condition_start_date").over(
        W.partitionBy("person_id", "provider_id").orderBy("condition_start_date"))) \
    .withColumn("days_to_next_visit", 
                F.datediff(F.col("next_visit"), F.col("condition_end_date"))) \
    .withColumn("is_readmission", 
                F.when((F.col("days_to_next_visit") >= 0) & (F.col("days_to_next_visit") <= 30), 1).otherwise(0))

quality_indicators = readmission_proxy \
    .groupBy("provider_id") \
    .agg(
        F.count("*").alias("total_episodes"),
        F.sum("is_readmission").alias("readmissions_30day"),
        F.avg("days_to_next_visit").alias("avg_days_between_visits"),
        F.avg(F.datediff(F.col("condition_end_date"), F.col("condition_start_date"))).alias("avg_episode_length")
    ) \
    .withColumn("readmission_rate", 
                F.when(F.col("total_episodes") > 0,
                      F.col("readmissions_30day") / F.col("total_episodes")).otherwise(0)) \
    .withColumn("processed_timestamp", F.lit(datetime.now().isoformat()))

silver_quality_indicators = quality_indicators

print(f"✓ Quality indicators: {silver_quality_indicators.count()} providers")
silver_quality_indicators.show(5, truncate=False)

silver_quality_indicators = add_layer_metadata(
    df=silver_quality_indicators,
    layer="silver",
    source_tables=["bronze_condition"],
    target_table="silver_quality_indicators"
)

# Write immediately!
silver_quality_indicators.write.mode("overwrite").parquet("./output/silver/silver_quality_indicators")
print(f"✓ QI written")

# Read back
silver_quality_indicators = spark.read.parquet("./output/silver/silver_quality_indicators")
print(f"  QI: {silver_quality_indicators.count()}")


SILVER 5: Provider Quality Indicators
✓ Quality indicators: 798 providers
+-----------+--------------+------------------+-----------------------+------------------+----------------+--------------------------+
|provider_id|total_episodes|readmissions_30day|avg_days_between_visits|avg_episode_length|readmission_rate|processed_timestamp       |
+-----------+--------------+------------------+-----------------------+------------------+----------------+--------------------------+
|74482      |1             |0                 |NULL                   |12.0              |0.0             |2025-11-23T22:24:39.220497|
|411979     |1             |0                 |NULL                   |6.0               |0.0             |2025-11-23T22:24:39.220497|
|111303     |1             |0                 |NULL                   |2.0               |0.0             |2025-11-23T22:24:39.220497|
|22096      |1             |0                 |NULL                   |7.0               |0.0             |2025-11-

25/11/23 22:24:39 ERROR ContextFactory: Query execution is null: can't emit event for executionId 353
25/11/23 22:24:39 ERROR ContextFactory: Query execution is null: can't emit event for executionId 353
25/11/23 22:24:39 ERROR ContextFactory: Query execution is null: can't emit event for executionId 353
25/11/23 22:24:39 ERROR ContextFactory: Query execution is null: can't emit event for executionId 353
25/11/23 22:24:39 ERROR ContextFactory: Query execution is null: can't emit event for executionId 353
25/11/23 22:24:39 ERROR ContextFactory: Query execution is null: can't emit event for executionId 353
25/11/23 22:24:39 ERROR ContextFactory: Query execution is null: can't emit event for executionId 354
25/11/23 22:24:39 ERROR ContextFactory: Query execution is null: can't emit event for executionId 354
25/11/23 22:24:39 ERROR ContextFactory: Query execution is null: can't emit event for executionId 354
25/11/23 22:24:39 ERROR ContextFactory: Query execution is null: can't emit event 

In [138]:
print("\n" + "="*60)
print("SILVER 6: Provider Care Coordination")
print("="*60)

patient_provider_count = bronze_condition \
    .select("person_id", "provider_id") \
    .union(bronze_procedure.select("person_id", "provider_id")) \
    .union(bronze_drug.select("person_id", "provider_id")) \
    .distinct() \
    .groupBy("person_id") \
    .agg(F.countDistinct("provider_id").alias("provider_count_per_patient"))

coordination_metrics = bronze_condition \
    .join(patient_provider_count, "person_id") \
    .groupBy(bronze_condition.provider_id) \
    .agg(
        F.avg("provider_count_per_patient").alias("avg_providers_per_patient"),
        F.max("provider_count_per_patient").alias("max_providers_per_patient"),
        F.countDistinct(bronze_condition.person_id).alias("patients_requiring_coordination")
    ) \
    .withColumn("coordination_complexity_score",
                F.col("avg_providers_per_patient") * F.col("patients_requiring_coordination") / 100) \
    .withColumn("processed_timestamp", F.lit(datetime.now().isoformat()))

silver_care_coordination = coordination_metrics

print(f"✓ Care coordination: {silver_care_coordination.count()} providers")
silver_care_coordination.show(5, truncate=False)

silver_care_coordination = add_layer_metadata(
    df=silver_care_coordination,
    layer="silver",
    source_tables=["bronze_condition", "bronze_procedure", "bronze_drug"],
    target_table="silver_care_coordination"
)

# Write immediately!
silver_care_coordination.write.mode("overwrite").parquet("./output/silver/silver_care_coordination")
print(f"✓ CC written")

# Read back
silver_care_coordination = spark.read.parquet("./output/silver/silver_care_coordination")
print(f"  CC: {silver_care_coordination.count()}")

25/11/23 22:24:39 ERROR ContextFactory: Query execution is null: can't emit event for executionId 358
25/11/23 22:24:39 ERROR ContextFactory: Query execution is null: can't emit event for executionId 358
25/11/23 22:24:39 ERROR ContextFactory: Query execution is null: can't emit event for executionId 358



SILVER 6: Provider Care Coordination
✓ Care coordination: 798 providers
+-----------+-------------------------+-------------------------+-------------------------------+-----------------------------+--------------------------+
|provider_id|avg_providers_per_patient|max_providers_per_patient|patients_requiring_coordination|coordination_complexity_score|processed_timestamp       |
+-----------+-------------------------+-------------------------+-------------------------------+-----------------------------+--------------------------+
|2304       |1.0                      |1                        |3                              |0.03                         |2025-11-23T22:24:39.523095|
|3683       |1.0                      |1                        |3                              |0.03                         |2025-11-23T22:24:39.523095|
|2875       |1.0                      |1                        |5                              |0.05                         |2025-11-23T22:24:39.52309

# STEP 4: Build Gold Layer (15 Vertical Transformations)

In [ ]:
print("\n" + "="*60)
print("GOLD 1: Provider Master Profile")
print("="*60)

os.makedirs("./output/gold", exist_ok=True)

gold_provider_master = silver_provider_demographics \
    .join(silver_provider_activity, "provider_id", "left") \
    .join(silver_patient_outcomes, "provider_id", "left") \
    .fillna(0)

# Drop duplicate processed_timestamp columns (keep only one)
columns_to_keep = [col for col in gold_provider_master.columns if col != "processed_timestamp"]
gold_provider_master = gold_provider_master.select(columns_to_keep)

# Now add a single processed_timestamp column
gold_provider_master = gold_provider_master \
    .withColumn("provider_age", F.lit(2024) - F.col("year_of_birth")) \
    .withColumn("patient_per_encounter_ratio",
                F.when(F.col("total_encounters") > 0,
                      F.col("total_unique_patients") / F.col("total_encounters")).otherwise(0)) \
    .withColumn("processed_timestamp", F.lit(datetime.now().isoformat()))

print(f"✓ Provider master profile: {gold_provider_master.count()} providers")
gold_provider_master.show(5, truncate=False)

gold_provider_master = add_layer_metadata(
    df=gold_provider_master,
    layer="gold",
    source_tables=["silver_provider_demographics", "silver_provider_activity", "silver_patient_outcomes"],
    target_table="gold_provider_master_profile"
)

# Write immediately!
gold_provider_master.write.mode("overwrite").parquet("./output/gold/gold_provider_master")
print(f"✓ PM written")

# Read back
gold_provider_master = spark.read.parquet("./output/gold/gold_provider_master")
print(f"  PM: {gold_provider_master.count()}")


GOLD 1: Provider Master Profile
✓ Provider master profile: 1000 providers
+-----------+-------------+----------+----+--------------------+----------------------+------------+-------------+-----------------+--------------+---------------------------+---------------+--------------------------+-----------------+----------+---------------------+------------+---------------+--------------------------+-----------------+----------------+---------------------+--------------+-----------------+---------------------------+--------------------+--------------+------------+---------------------------+--------------------------+
|provider_id|provider_name|npi       |dea |specialty_concept_id|specialty_source_value|care_site_id|year_of_birth|gender_concept_id|care_site_name|place_of_service_concept_id|procedure_count|unique_patients_procedures|procedure_variety|drug_count|unique_patients_drugs|drug_variety|condition_count|unique_patients_conditions|condition_variety|total_encounters|total_unique_pati

25/11/23 22:24:40 ERROR ContextFactory: Query execution is null: can't emit event for executionId 361
25/11/23 22:24:40 ERROR ContextFactory: Query execution is null: can't emit event for executionId 361
25/11/23 22:24:40 ERROR ContextFactory: Query execution is null: can't emit event for executionId 361
25/11/23 22:24:40 ERROR ContextFactory: Query execution is null: can't emit event for executionId 361
25/11/23 22:24:40 ERROR ContextFactory: Query execution is null: can't emit event for executionId 361
25/11/23 22:24:40 ERROR ContextFactory: Query execution is null: can't emit event for executionId 361
25/11/23 22:24:40 ERROR ContextFactory: Query execution is null: can't emit event for executionId 362
25/11/23 22:24:40 ERROR ContextFactory: Query execution is null: can't emit event for executionId 362
25/11/23 22:24:40 ERROR ContextFactory: Query execution is null: can't emit event for executionId 362
25/11/23 22:24:40 ERROR ContextFactory: Query execution is null: can't emit event 

In [140]:
print("\n" + "="*60)
print("GOLD 2: Clinical Volume Metrics")
print("="*60)

volume_window = W.orderBy(F.col("total_encounters").desc())

gold_volume_metrics = silver_provider_activity \
    .withColumn("volume_rank", F.row_number().over(volume_window)) \
    .withColumn("volume_percentile", F.percent_rank().over(volume_window)) \
    .withColumn("is_high_volume", F.when(F.col("volume_percentile") >= 0.75, 1).otherwise(0)) \
    .withColumn("procedure_ratio", 
                F.when(F.col("total_encounters") > 0,
                      F.col("procedure_count") / F.col("total_encounters")).otherwise(0)) \
    .withColumn("drug_ratio",
                F.when(F.col("total_encounters") > 0,
                      F.col("drug_count") / F.col("total_encounters")).otherwise(0)) \
    .withColumn("processed_timestamp", F.lit(datetime.now().isoformat()))

# Write immediately!
gold_volume_metrics.write.mode("overwrite").parquet("./output/gold/gold_volume_metrics")
print(f"✓ VM written")

# Read back
gold_volume_metrics = spark.read.parquet("./output/gold/gold_volume_metrics")
print(f"  VM: {gold_volume_metrics.count()}")

print(f"✓ Volume metrics: {gold_volume_metrics.count()} providers")
gold_volume_metrics.show(5, truncate=False)

25/11/23 22:24:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/23 22:24:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/23 22:24:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/23 22:24:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/23 22:24:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.



GOLD 2: Clinical Volume Metrics
✓ VM written
  VM: 2408


25/11/23 22:24:40 ERROR ContextFactory: Query execution is null: can't emit event for executionId 366
25/11/23 22:24:40 ERROR ContextFactory: Query execution is null: can't emit event for executionId 366
25/11/23 22:24:40 ERROR ContextFactory: Query execution is null: can't emit event for executionId 366


✓ Volume metrics: 2408 providers
+-----------+---------------+--------------------------+-----------------+----------+---------------------+------------+---------------+--------------------------+-----------------+----------------+---------------------+--------------------------+-----------+--------------------+--------------+-------------------+----------+
|provider_id|procedure_count|unique_patients_procedures|procedure_variety|drug_count|unique_patients_drugs|drug_variety|condition_count|unique_patients_conditions|condition_variety|total_encounters|total_unique_patients|processed_timestamp       |volume_rank|volume_percentile   |is_high_volume|procedure_ratio    |drug_ratio|
+-----------+---------------+--------------------------+-----------------+----------+---------------------+------------+---------------+--------------------------+-----------------+----------------+---------------------+--------------------------+-----------+--------------------+--------------+------------------

In [141]:
print("\n" + "="*60)
print("GOLD 3: Clinical Diversity Index")
print("="*60)

gold_clinical_diversity = silver_provider_activity \
    .withColumn("diversity_score",
                (F.col("procedure_variety") + F.col("drug_variety") + F.col("condition_variety")) / 3.0) \
    .withColumn("specialization_index",
                F.when(F.col("diversity_score") > 0,
                      1.0 / F.col("diversity_score")).otherwise(0)) \
    .withColumn("is_specialist", F.when(F.col("specialization_index") > 0.1, 1).otherwise(0)) \
    .withColumn("is_generalist", F.when(F.col("diversity_score") > 50, 1).otherwise(0)) \
    .withColumn("processed_timestamp", F.lit(datetime.now().isoformat()))

# Write immediately!
gold_clinical_diversity.write.mode("overwrite").parquet("./output/gold/gold_clinical_diversity")
print(f"✓ CD written")

# Read back
gold_clinical_diversity = spark.read.parquet("./output/gold/gold_clinical_diversity")
print(f"  CD: {gold_clinical_diversity.count()}")

print(f"✓ Clinical diversity: {gold_clinical_diversity.count()} providers")
gold_clinical_diversity.show(5, truncate=False)

25/11/23 22:24:40 ERROR ContextFactory: Query execution is null: can't emit event for executionId 368
25/11/23 22:24:40 ERROR ContextFactory: Query execution is null: can't emit event for executionId 368
25/11/23 22:24:40 ERROR ContextFactory: Query execution is null: can't emit event for executionId 368
25/11/23 22:24:40 ERROR ContextFactory: Query execution is null: can't emit event for executionId 369
25/11/23 22:24:40 ERROR ContextFactory: Query execution is null: can't emit event for executionId 369
25/11/23 22:24:40 ERROR ContextFactory: Query execution is null: can't emit event for executionId 369
25/11/23 22:24:40 ERROR ContextFactory: Query execution is null: can't emit event for executionId 370
25/11/23 22:24:40 ERROR ContextFactory: Query execution is null: can't emit event for executionId 370



GOLD 3: Clinical Diversity Index
✓ CD written
  CD: 2408
✓ Clinical diversity: 2408 providers
+-----------+---------------+--------------------------+-----------------+----------+---------------------+------------+---------------+--------------------------+-----------------+----------------+---------------------+--------------------------+------------------+--------------------+-------------+-------------+
|provider_id|procedure_count|unique_patients_procedures|procedure_variety|drug_count|unique_patients_drugs|drug_variety|condition_count|unique_patients_conditions|condition_variety|total_encounters|total_unique_patients|processed_timestamp       |diversity_score   |specialization_index|is_specialist|is_generalist|
+-----------+---------------+--------------------------+-----------------+----------+---------------------+------------+---------------+--------------------------+-----------------+----------------+---------------------+--------------------------+------------------+-------

25/11/23 22:24:40 ERROR ContextFactory: Query execution is null: can't emit event for executionId 371
25/11/23 22:24:40 ERROR ContextFactory: Query execution is null: can't emit event for executionId 371
25/11/23 22:24:40 ERROR ContextFactory: Query execution is null: can't emit event for executionId 372
25/11/23 22:24:40 ERROR ContextFactory: Query execution is null: can't emit event for executionId 372
25/11/23 22:24:40 ERROR ContextFactory: Query execution is null: can't emit event for executionId 372
25/11/23 22:24:40 ERROR ContextFactory: Query execution is null: can't emit event for executionId 373
25/11/23 22:24:40 ERROR ContextFactory: Query execution is null: can't emit event for executionId 373
25/11/23 22:24:40 ERROR ContextFactory: Query execution is null: can't emit event for executionId 373


In [142]:
print("\n" + "="*60)
print("GOLD 4: Outcome Performance Tiers")
print("="*60)

outcome_window = W.orderBy(F.col("mortality_rate"))

gold_outcome_tiers = silver_patient_outcomes \
    .withColumn("mortality_percentile", F.percent_rank().over(outcome_window)) \
    .withColumn("outcome_tier",
                F.when(F.col("mortality_percentile") <= 0.25, "Excellent")
                 .when(F.col("mortality_percentile") <= 0.50, "Good")
                 .when(F.col("mortality_percentile") <= 0.75, "Fair")
                 .otherwise("Needs Improvement")) \
    .withColumn("complexity_adjusted_mortality",
                F.when(F.col("condition_complexity") > 0,
                      F.col("mortality_rate") / F.col("condition_complexity")).otherwise(0)) \
    .withColumn("processed_timestamp", F.lit(datetime.now().isoformat()))

# Write immediately!
gold_outcome_tiers.write.mode("overwrite").parquet("./output/gold/gold_outcome_tiers")
print(f"✓ OT written")

# Read back
gold_outcome_tiers = spark.read.parquet("./output/gold/gold_outcome_tiers")
print(f"  OT: {gold_outcome_tiers.count()}")

print(f"✓ Outcome tiers: {gold_outcome_tiers.count()} providers")
gold_outcome_tiers.show(5, truncate=False)


GOLD 4: Outcome Performance Tiers


25/11/23 22:24:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/23 22:24:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/23 22:24:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/23 22:24:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/23 22:24:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


✓ OT written
  OT: 2
✓ Outcome tiers: 2 providers
+-----------+--------------+-----------------+---------------------------+--------------------+--------------------+--------------------------+--------------------+-----------------+-----------------------------+
|provider_id|total_patients|deceased_patients|avg_condition_duration_days|condition_complexity|mortality_rate      |processed_timestamp       |mortality_percentile|outcome_tier     |complexity_adjusted_mortality|
+-----------+--------------+-----------------+---------------------------+--------------------+--------------------+--------------------------+--------------------+-----------------+-----------------------------+
|9069       |1             |0                |3.0                        |1                   |0.0                 |2025-11-23T22:24:40.674844|0.0                 |Excellent        |0.0                          |
|NULL       |999           |1                |NULL                       |0                   |0.0

In [143]:
print("\n" + "="*60)
print("GOLD 5: Network Connectivity Score")
print("="*60)

network_window = W.orderBy(F.col("network_centrality").desc())

gold_network_connectivity = silver_referral_network \
    .withColumn("network_rank", F.row_number().over(network_window)) \
    .withColumn("network_percentile", F.percent_rank().over(network_window)) \
    .withColumn("is_hub_provider", F.when(F.col("network_percentile") >= 0.90, 1).otherwise(0)) \
    .withColumn("referral_balance",
                F.abs(F.col("outbound_referral_count") - F.col("inbound_referral_count"))) \
    .withColumn("is_balanced_network", F.when(F.col("referral_balance") <= 2, 1).otherwise(0)) \
    .withColumn("processed_timestamp", F.lit(datetime.now().isoformat()))

# Write immediately!
gold_network_connectivity.write.mode("overwrite").parquet("./output/gold/gold_network_connectivity")
print(f"✓ NC written")

# Read back
gold_network_connectivity = spark.read.parquet("./output/gold/gold_network_connectivity")
print(f"  NC: {gold_network_connectivity.count()}")

print(f"✓ Network connectivity: {gold_network_connectivity.count()} providers")
gold_network_connectivity.show(5, truncate=False)


GOLD 5: Network Connectivity Score
✓ NC written
  NC: 6
✓ Network connectivity: 6 providers
+-----------+-----------------------+---------------------------+----------------------+--------------------------+------------------+--------------------------+------------+------------------+---------------+----------------+-------------------+
|provider_id|outbound_referral_count|total_patients_referred_out|inbound_referral_count|total_patients_referred_in|network_centrality|processed_timestamp       |network_rank|network_percentile|is_hub_provider|referral_balance|is_balanced_network|
+-----------+-----------------------+---------------------------+----------------------+--------------------------+------------------+--------------------------+------------+------------------+---------------+----------------+-------------------+
|2317       |0                      |0                          |1                     |1                         |1                 |2025-11-23T22:24:40.876913|1    

25/11/23 22:24:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/23 22:24:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/23 22:24:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/23 22:24:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/23 22:24:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/23 22:24:40 ERROR ContextFactory: Query execution is null: can't emit event for executionId 376
25/11/23 22:24:40 ERROR ContextFactory: Query execution is null: can't em

In [144]:
print("\n" + "="*60)
print("GOLD 6: Quality Composite Subscores")
print("="*60)

quality_window = W.orderBy(F.col("readmission_rate"))

gold_quality_subscores = silver_quality_indicators \
    .withColumn("readmission_percentile", F.percent_rank().over(quality_window)) \
    .withColumn("readmission_score", (1 - F.col("readmission_percentile")) * 100) \
    .withColumn("episode_efficiency_score",
                F.when(F.col("avg_episode_length") > 0,
                      100.0 / F.col("avg_episode_length")).otherwise(0)) \
    .withColumn("continuity_score",
                F.when(F.col("avg_days_between_visits") > 0,
                      F.least(F.lit(100), 100.0 / F.col("avg_days_between_visits"))).otherwise(0)) \
    .withColumn("processed_timestamp", F.lit(datetime.now().isoformat()))

# Write immediately!
gold_quality_subscores.write.mode("overwrite").parquet("./output/gold/gold_quality_subscores")
print(f"✓ QS written")

# Read back
gold_quality_subscores = spark.read.parquet("./output/gold/gold_quality_subscores")
print(f"  QS: {gold_quality_subscores.count()}")

print(f"✓ Quality subscores: {gold_quality_subscores.count()} providers")
gold_quality_subscores.show(5, truncate=False)


GOLD 6: Quality Composite Subscores


25/11/23 22:24:41 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/23 22:24:41 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/23 22:24:41 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


✓ QS written
  QS: 798


25/11/23 22:24:41 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/23 22:24:41 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/23 22:24:41 ERROR ContextFactory: Query execution is null: can't emit event for executionId 380
25/11/23 22:24:41 ERROR ContextFactory: Query execution is null: can't emit event for executionId 380
25/11/23 22:24:41 ERROR ContextFactory: Query execution is null: can't emit event for executionId 380
25/11/23 22:24:41 ERROR ContextFactory: Query execution is null: can't emit event for executionId 381
25/11/23 22:24:41 ERROR ContextFactory: Query execution is null: can't emit event for executionId 381
25/11/23 22:24:41 ERROR ContextFactory: Query execution is null: can't emit event for executionId 381
25/11/23 22:24:41 ERROR ContextFactory: Query execution is

✓ Quality subscores: 798 providers
+-----------+--------------+------------------+-----------------------+------------------+----------------+--------------------------+----------------------+-----------------+------------------------+----------------+
|provider_id|total_episodes|readmissions_30day|avg_days_between_visits|avg_episode_length|readmission_rate|processed_timestamp       |readmission_percentile|readmission_score|episode_efficiency_score|continuity_score|
+-----------+--------------+------------------+-----------------------+------------------+----------------+--------------------------+----------------------+-----------------+------------------------+----------------+
|74482      |1             |0                 |NULL                   |12.0              |0.0             |2025-11-23T22:24:41.074714|0.0                   |100.0            |8.333333333333334       |0.0             |
|411979     |1             |0                 |NULL                   |6.0               |0.0

25/11/23 22:24:41 ERROR ContextFactory: Query execution is null: can't emit event for executionId 384
25/11/23 22:24:41 ERROR ContextFactory: Query execution is null: can't emit event for executionId 384
25/11/23 22:24:41 ERROR ContextFactory: Query execution is null: can't emit event for executionId 384
25/11/23 22:24:41 ERROR ContextFactory: Query execution is null: can't emit event for executionId 385
25/11/23 22:24:41 ERROR ContextFactory: Query execution is null: can't emit event for executionId 385
25/11/23 22:24:41 ERROR ContextFactory: Query execution is null: can't emit event for executionId 385


In [145]:
print("\n" + "="*60)
print("GOLD 7: Coordination Effectiveness")
print("="*60)

coord_window = W.orderBy(F.col("coordination_complexity_score").desc())

gold_coordination_effectiveness = silver_care_coordination \
    .withColumn("coordination_rank", F.row_number().over(coord_window)) \
    .withColumn("coordination_percentile", F.percent_rank().over(coord_window)) \
    .withColumn("is_coordination_specialist", 
                F.when(F.col("coordination_percentile") >= 0.75, 1).otherwise(0)) \
    .withColumn("collaboration_intensity",
                F.col("avg_providers_per_patient") * F.col("patients_requiring_coordination")) \
    .withColumn("processed_timestamp", F.lit(datetime.now().isoformat()))

# Write immediately!
gold_coordination_effectiveness.write.mode("overwrite").parquet("./output/gold/gold_coordination_effectiveness")
print(f"✓ CE written")

# Read back
gold_coordination_effectiveness = spark.read.parquet("./output/gold/gold_coordination_effectiveness")
print(f"  CE: {gold_coordination_effectiveness.count()}")

print(f"✓ Coordination effectiveness: {gold_coordination_effectiveness.count()} providers")
gold_coordination_effectiveness.show(5, truncate=False)


GOLD 7: Coordination Effectiveness
✓ CE written
  CE: 798
✓ Coordination effectiveness: 798 providers


25/11/23 22:24:41 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/23 22:24:41 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/23 22:24:41 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/23 22:24:41 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/23 22:24:41 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+-----------+-------------------------+-------------------------+-------------------------------+-----------------------------+--------------------------+-----------------+-----------------------+--------------------------+-----------------------+
|provider_id|avg_providers_per_patient|max_providers_per_patient|patients_requiring_coordination|coordination_complexity_score|processed_timestamp       |coordination_rank|coordination_percentile|is_coordination_specialist|collaboration_intensity|
+-----------+-------------------------+-------------------------+-------------------------------+-----------------------------+--------------------------+-----------------+-----------------------+--------------------------+-----------------------+
|8823       |1.0                      |1                        |7                              |0.07                         |2025-11-23T22:24:41.318853|1                |0.0                    |0                         |7.0                    |
|34619  

In [146]:
print("\n" + "="*60)
print("GOLD 8: Provider Efficiency Index")
print("="*60)

gold_efficiency_index = gold_provider_master \
    .join(gold_volume_metrics.select("provider_id", "volume_percentile", "procedure_ratio"), "provider_id") \
    .join(gold_outcome_tiers.select("provider_id", "mortality_percentile"), "provider_id") \
    

# Drop duplicate processed_timestamp columns
columns_to_keep = [col for col in gold_efficiency_index.columns if col != "processed_timestamp"]
gold_efficiency_index = gold_efficiency_index.select(columns_to_keep)

# Now add one processed_timestamp
gold_efficiency_index = gold_efficiency_index \
    .withColumn("efficiency_score",
                (F.col("volume_percentile") * 0.4 + 
                 (1 - F.col("mortality_percentile")) * 0.4 +
                 F.col("patient_per_encounter_ratio") * 0.2) * 100) \
    .withColumn("efficiency_grade",
                F.when(F.col("efficiency_score") >= 80, "A")
                 .when(F.col("efficiency_score") >= 60, "B")
                 .when(F.col("efficiency_score") >= 40, "C")
                 .otherwise("D")) \
    .withColumn("processed_timestamp", F.lit(datetime.now().isoformat()))

# Write immediately!
gold_efficiency_index.write.mode("overwrite").parquet("./output/gold/gold_efficiency_index")
print(f"✓ EI written")

# Read back
gold_efficiency_index = spark.read.parquet("./output/gold/gold_efficiency_index")
print(f"  EI: {gold_efficiency_index.count()}")

print(f"✓ Efficiency index: {gold_efficiency_index.count()} providers")
gold_efficiency_index.show(5, truncate=False)


GOLD 8: Provider Efficiency Index
✓ EI written


25/11/23 22:24:41 ERROR ContextFactory: Query execution is null: can't emit event for executionId 388
25/11/23 22:24:41 ERROR ContextFactory: Query execution is null: can't emit event for executionId 388
25/11/23 22:24:41 ERROR ContextFactory: Query execution is null: can't emit event for executionId 388
25/11/23 22:24:41 ERROR ContextFactory: Query execution is null: can't emit event for executionId 389
25/11/23 22:24:41 ERROR ContextFactory: Query execution is null: can't emit event for executionId 389
25/11/23 22:24:41 ERROR ContextFactory: Query execution is null: can't emit event for executionId 389
25/11/23 22:24:41 ERROR ContextFactory: Query execution is null: can't emit event for executionId 390
25/11/23 22:24:41 ERROR ContextFactory: Query execution is null: can't emit event for executionId 390


  EI: 0
✓ Efficiency index: 0 providers
+-----------+-------------+---+---+--------------------+----------------------+------------+-------------+-----------------+--------------+---------------------------+---------------+--------------------------+-----------------+----------+---------------------+------------+---------------+--------------------------+-----------------+----------------+---------------------+--------------+-----------------+---------------------------+--------------------+--------------+------------+---------------------------+-----------------+---------------+--------------------+----------------+----------------+-------------------+
|provider_id|provider_name|npi|dea|specialty_concept_id|specialty_source_value|care_site_id|year_of_birth|gender_concept_id|care_site_name|place_of_service_concept_id|procedure_count|unique_patients_procedures|procedure_variety|drug_count|unique_patients_drugs|drug_variety|condition_count|unique_patients_conditions|condition_variety|tot

In [147]:
print("\n" + "="*60)
print("GOLD 9: Network Influence Metrics")
print("="*60)

gold_network_influence = gold_network_connectivity \
    .withColumn("influence_score",
                (F.col("network_centrality") * 0.5 +
                 F.col("total_patients_referred_out") * 0.3 +
                 F.col("total_patients_referred_in") * 0.2)) \
    .withColumn("referral_direction",
                F.when(F.col("outbound_referral_count") > F.col("inbound_referral_count"), "Outbound")
                 .when(F.col("inbound_referral_count") > F.col("outbound_referral_count"), "Inbound")
                 .otherwise("Balanced")) \
    .withColumn("network_role",
                F.when(F.col("is_hub_provider") == 1, "Hub")
                 .when(F.col("is_balanced_network") == 1, "Connector")
                 .otherwise("Peripheral")) \
    .withColumn("processed_timestamp", F.lit(datetime.now().isoformat()))

# Write immediately!
gold_network_influence.write.mode("overwrite").parquet("./output/gold/gold_network_influence")
print(f"✓ NI written")

# Read back
gold_network_influence = spark.read.parquet("./output/gold/gold_network_influence")
print(f"  NI: {gold_network_influence.count()}")

print(f"✓ Network influence: {gold_network_influence.count()} providers")
gold_network_influence.show(5, truncate=False)


GOLD 9: Network Influence Metrics
✓ NI written
  NI: 6
✓ Network influence: 6 providers
+-----------+-----------------------+---------------------------+----------------------+--------------------------+------------------+--------------------------+------------+------------------+---------------+----------------+-------------------+---------------+------------------+------------+
|provider_id|outbound_referral_count|total_patients_referred_out|inbound_referral_count|total_patients_referred_in|network_centrality|processed_timestamp       |network_rank|network_percentile|is_hub_provider|referral_balance|is_balanced_network|influence_score|referral_direction|network_role|
+-----------+-----------------------+---------------------------+----------------------+--------------------------+------------------+--------------------------+------------+------------------+---------------+----------------+-------------------+---------------+------------------+------------+
|2317       |0            

In [148]:
print("\n" + "="*60)
print("GOLD 10: Quality-Adjusted Performance")
print("="*60)

gold_quality_adjusted = gold_quality_subscores \
    .withColumn("quality_composite",
                (F.col("readmission_score") * 0.4 +
                 F.col("episode_efficiency_score") * 0.3 +
                 F.col("continuity_score") * 0.3)) \
    .withColumn("quality_tier",
                F.when(F.col("quality_composite") >= 75, "Tier 1")
                 .when(F.col("quality_composite") >= 50, "Tier 2")
                 .when(F.col("quality_composite") >= 25, "Tier 3")
                 .otherwise("Tier 4")) \
    .withColumn("is_quality_leader", F.when(F.col("quality_composite") >= 75, 1).otherwise(0)) \
    .withColumn("processed_timestamp", F.lit(datetime.now().isoformat()))

# Write immediately!
gold_quality_adjusted.write.mode("overwrite").parquet("./output/gold/gold_quality_adjusted")
print(f"✓ QA written")

# Read back
gold_quality_adjusted = spark.read.parquet("./output/gold/gold_quality_adjusted")
print(f"  QA: {gold_quality_adjusted.count()}")

print(f"✓ Quality-adjusted performance: {gold_quality_adjusted.count()} providers")
gold_quality_adjusted.show(5, truncate=False)


GOLD 10: Quality-Adjusted Performance
✓ QA written
  QA: 798
✓ Quality-adjusted performance: 798 providers
+-----------+--------------+------------------+-----------------------+------------------+----------------+--------------------------+----------------------+-----------------+------------------------+----------------+------------------+------------+-----------------+
|provider_id|total_episodes|readmissions_30day|avg_days_between_visits|avg_episode_length|readmission_rate|processed_timestamp       |readmission_percentile|readmission_score|episode_efficiency_score|continuity_score|quality_composite |quality_tier|is_quality_leader|
+-----------+--------------+------------------+-----------------------+------------------+----------------+--------------------------+----------------------+-----------------+------------------------+----------------+------------------+------------+-----------------+
|74482      |1             |0                 |NULL                   |12.0             

25/11/23 22:24:42 ERROR ContextFactory: Query execution is null: can't emit event for executionId 392
25/11/23 22:24:42 ERROR ContextFactory: Query execution is null: can't emit event for executionId 392
25/11/23 22:24:42 ERROR ContextFactory: Query execution is null: can't emit event for executionId 392
25/11/23 22:24:42 ERROR ContextFactory: Query execution is null: can't emit event for executionId 393
25/11/23 22:24:42 ERROR ContextFactory: Query execution is null: can't emit event for executionId 393
25/11/23 22:24:42 ERROR ContextFactory: Query execution is null: can't emit event for executionId 393
25/11/23 22:24:42 ERROR ContextFactory: Query execution is null: can't emit event for executionId 394
25/11/23 22:24:42 ERROR ContextFactory: Query execution is null: can't emit event for executionId 394
25/11/23 22:24:42 ERROR ContextFactory: Query execution is null: can't emit event for executionId 395
25/11/23 22:24:42 ERROR ContextFactory: Query execution is null: can't emit event 

In [149]:
print("\n" + "="*60)
print("GOLD 11: Collaboration Network Density")
print("="*60)

gold_collaboration_density = gold_coordination_effectiveness \
    .withColumn("network_density_score",
                F.col("coordination_complexity_score") * F.col("avg_providers_per_patient")) \
    .withColumn("collaboration_type",
                F.when(F.col("is_coordination_specialist") == 1, "High Collaboration")
                 .when(F.col("avg_providers_per_patient") >= 2, "Moderate Collaboration")
                 .otherwise("Low Collaboration")) \
    .withColumn("team_based_care_indicator",
                F.when(F.col("avg_providers_per_patient") >= 3, 1).otherwise(0)) \
    .withColumn("processed_timestamp", F.lit(datetime.now().isoformat()))

# Write immediately!
gold_collaboration_density.write.mode("overwrite").parquet("./output/gold/gold_collaboration_density")
print(f"✓ CD written")

# Read back
gold_collaboration_density = spark.read.parquet("./output/gold/gold_collaboration_density")
print(f"  CD: {gold_collaboration_density.count()}")

print(f"✓ Collaboration density: {gold_collaboration_density.count()} providers")
gold_collaboration_density.show(5, truncate=False)


GOLD 11: Collaboration Network Density


25/11/23 22:24:42 ERROR ContextFactory: Query execution is null: can't emit event for executionId 400
25/11/23 22:24:42 ERROR ContextFactory: Query execution is null: can't emit event for executionId 400
25/11/23 22:24:42 ERROR ContextFactory: Query execution is null: can't emit event for executionId 400
25/11/23 22:24:42 ERROR ContextFactory: Query execution is null: can't emit event for executionId 401
25/11/23 22:24:42 ERROR ContextFactory: Query execution is null: can't emit event for executionId 401
25/11/23 22:24:42 ERROR ContextFactory: Query execution is null: can't emit event for executionId 401
25/11/23 22:24:42 ERROR ContextFactory: Query execution is null: can't emit event for executionId 402
25/11/23 22:24:42 ERROR ContextFactory: Query execution is null: can't emit event for executionId 402


✓ CD written
  CD: 798
✓ Collaboration density: 798 providers
+-----------+-------------------------+-------------------------+-------------------------------+-----------------------------+--------------------------+-----------------+-----------------------+--------------------------+-----------------------+---------------------+------------------+-------------------------+
|provider_id|avg_providers_per_patient|max_providers_per_patient|patients_requiring_coordination|coordination_complexity_score|processed_timestamp       |coordination_rank|coordination_percentile|is_coordination_specialist|collaboration_intensity|network_density_score|collaboration_type|team_based_care_indicator|
+-----------+-------------------------+-------------------------+-------------------------------+-----------------------------+--------------------------+-----------------+-----------------------+--------------------------+-----------------------+---------------------+------------------+--------------------

In [150]:
print("\n" + "="*60)
print("GOLD 12: Outcome Attribution Scores")
print("="*60)

gold_outcome_attribution = gold_outcome_tiers \
    .join(gold_volume_metrics.select("provider_id", "total_encounters", "total_unique_patients"), "provider_id")


# Drop duplicate processed_timestamp columns
columns_to_keep = [col for col in gold_outcome_attribution.columns if col != "processed_timestamp"]
gold_outcome_attribution = gold_outcome_attribution.select(columns_to_keep)

# Now add one processed_timestamp
gold_outcome_attribution = gold_outcome_attribution \
    .withColumn("attribution_weight",
                F.col("total_unique_patients") / (F.col("avg_condition_duration_days") + 1)) \
    .withColumn("outcome_contribution_score",
                (1 - F.col("mortality_rate")) * F.col("attribution_weight") * 100) \
    .withColumn("complexity_adjusted_attribution",
                F.col("outcome_contribution_score") / (F.col("condition_complexity") + 1)) \
    .withColumn("processed_timestamp", F.lit(datetime.now().isoformat()))

# Write immediately!
gold_outcome_attribution.write.mode("overwrite").parquet("./output/gold/gold_outcome_attribution")
print(f"✓ OA written")

# Read back
gold_outcome_attribution = spark.read.parquet("./output/gold/gold_outcome_attribution")
print(f"  OA: {gold_outcome_attribution.count()}")


print(f"✓ Outcome attribution: {gold_outcome_attribution.count()} providers")
gold_outcome_attribution.show(5, truncate=False)


GOLD 12: Outcome Attribution Scores


25/11/23 22:24:42 ERROR ContextFactory: Query execution is null: can't emit event for executionId 404
25/11/23 22:24:42 ERROR ContextFactory: Query execution is null: can't emit event for executionId 404
25/11/23 22:24:42 ERROR ContextFactory: Query execution is null: can't emit event for executionId 404
25/11/23 22:24:42 ERROR ContextFactory: Query execution is null: can't emit event for executionId 405
25/11/23 22:24:42 ERROR ContextFactory: Query execution is null: can't emit event for executionId 405
25/11/23 22:24:42 ERROR ContextFactory: Query execution is null: can't emit event for executionId 405
25/11/23 22:24:42 ERROR ContextFactory: Query execution is null: can't emit event for executionId 406
25/11/23 22:24:42 ERROR ContextFactory: Query execution is null: can't emit event for executionId 406


✓ OA written
  OA: 1
✓ Outcome attribution: 1 providers
+-----------+--------------+-----------------+---------------------------+--------------------+--------------+--------------------+------------+-----------------------------+----------------+---------------------+------------------+--------------------------+-------------------------------+--------------------------+
|provider_id|total_patients|deceased_patients|avg_condition_duration_days|condition_complexity|mortality_rate|mortality_percentile|outcome_tier|complexity_adjusted_mortality|total_encounters|total_unique_patients|attribution_weight|outcome_contribution_score|complexity_adjusted_attribution|processed_timestamp       |
+-----------+--------------+-----------------+---------------------------+--------------------+--------------+--------------------+------------+-----------------------------+----------------+---------------------+------------------+--------------------------+-------------------------------+---------------

In [151]:
print("\n" + "="*60)
print("GOLD 13: Network Efficiency Ratio")
print("="*60)

gold_network_efficiency = gold_network_influence \
    .join(gold_quality_adjusted.select("provider_id", "quality_composite"), "provider_id")

# Drop duplicate processed_timestamp columns
columns_to_keep = [col for col in gold_network_efficiency.columns if col != "processed_timestamp"]
gold_network_efficiency = gold_network_efficiency.select(columns_to_keep)

# Now add one processed_timestamp
gold_network_efficiency = gold_network_efficiency \
    .withColumn("efficiency_ratio",
                F.when(F.col("network_centrality") > 0,
                      F.col("quality_composite") / F.col("network_centrality")).otherwise(0)) \
    .withColumn("quality_per_referral",
                F.when((F.col("total_patients_referred_out") + F.col("total_patients_referred_in")) > 0,
                      F.col("quality_composite") / (F.col("total_patients_referred_out") + F.col("total_patients_referred_in")))
                 .otherwise(0)) \
    .withColumn("is_efficient_network_provider",
                F.when(F.col("efficiency_ratio") > F.lit(0).cast("double"), 1).otherwise(0)) \
    .withColumn("processed_timestamp", F.lit(datetime.now().isoformat()))

# Write immediately!
gold_network_efficiency.write.mode("overwrite").parquet("./output/gold/gold_network_efficiency")
print(f"✓ NE written")

# Read back
gold_network_efficiency = spark.read.parquet("./output/gold/gold_network_efficiency")
print(f"  NE: {gold_network_efficiency.count()}")

print(f"✓ Network efficiency ratio: {gold_network_efficiency.count()} providers")
gold_network_efficiency.show(5, truncate=False)


GOLD 13: Network Efficiency Ratio
✓ NE written
  NE: 2
✓ Network efficiency ratio: 2 providers
+-----------+-----------------------+---------------------------+----------------------+--------------------------+------------------+------------+------------------+---------------+----------------+-------------------+---------------+------------------+------------+-----------------+----------------+--------------------+-----------------------------+--------------------------+
|provider_id|outbound_referral_count|total_patients_referred_out|inbound_referral_count|total_patients_referred_in|network_centrality|network_rank|network_percentile|is_hub_provider|referral_balance|is_balanced_network|influence_score|referral_direction|network_role|quality_composite|efficiency_ratio|quality_per_referral|is_efficient_network_provider|processed_timestamp       |
+-----------+-----------------------+---------------------------+----------------------+--------------------------+------------------+--------

25/11/23 22:24:42 ERROR ContextFactory: Query execution is null: can't emit event for executionId 408
25/11/23 22:24:42 ERROR ContextFactory: Query execution is null: can't emit event for executionId 408
25/11/23 22:24:42 ERROR ContextFactory: Query execution is null: can't emit event for executionId 408
25/11/23 22:24:42 ERROR ContextFactory: Query execution is null: can't emit event for executionId 409
25/11/23 22:24:42 ERROR ContextFactory: Query execution is null: can't emit event for executionId 409
25/11/23 22:24:42 ERROR ContextFactory: Query execution is null: can't emit event for executionId 409
25/11/23 22:24:42 ERROR ContextFactory: Query execution is null: can't emit event for executionId 410
25/11/23 22:24:42 ERROR ContextFactory: Query execution is null: can't emit event for executionId 410


In [152]:
print("\n" + "="*60)
print("GOLD 14: Comprehensive Provider Scorecard")
print("="*60)

gold_comprehensive_scorecard = gold_provider_master \
    .join(gold_efficiency_index.select("provider_id", "efficiency_score", "efficiency_grade"), "provider_id") \
    .join(gold_quality_adjusted.select("provider_id", "quality_composite", "quality_tier"), "provider_id") \
    .join(gold_network_influence.select("provider_id", "influence_score", "network_role"), "provider_id") \
    .join(gold_collaboration_density.select("provider_id", "network_density_score", "collaboration_type"), "provider_id")

# Drop duplicate processed_timestamp columns
columns_to_keep = [col for col in gold_comprehensive_scorecard.columns if col != "processed_timestamp"]
gold_comprehensive_scorecard = gold_comprehensive_scorecard.select(columns_to_keep)

# Now add one processed_timestamp
gold_comprehensive_scorecard = gold_comprehensive_scorecard \
    .withColumn("overall_performance_score",
                (F.col("efficiency_score") * 0.3 +
                 F.col("quality_composite") * 0.3 +
                 F.col("influence_score") * 0.2 +
                 F.col("network_density_score") * 0.2)) \
    .withColumn("performance_category",
                F.when(F.col("overall_performance_score") >= 75, "Exceptional")
                 .when(F.col("overall_performance_score") >= 50, "Strong")
                 .when(F.col("overall_performance_score") >= 25, "Adequate")
                 .otherwise("Developing")) \
    .withColumn("processed_timestamp", F.lit(datetime.now().isoformat()))

# Write immediately!
gold_comprehensive_scorecard.write.mode("overwrite").parquet("./output/gold/gold_comprehensive_scorecard")
print(f"✓ CS written")

# Read back
gold_comprehensive_scorecard = spark.read.parquet("./output/gold/gold_comprehensive_scorecard")
print(f"  CS: {gold_comprehensive_scorecard.count()}")

print(f"✓ Comprehensive scorecard: {gold_comprehensive_scorecard.count()} providers")
gold_comprehensive_scorecard.show(5, truncate=False)


GOLD 14: Comprehensive Provider Scorecard
✓ CS written
  CS: 0
✓ Comprehensive scorecard: 0 providers
+-----------+-------------+---+---+--------------------+----------------------+------------+-------------+-----------------+--------------+---------------------------+---------------+--------------------------+-----------------+----------+---------------------+------------+---------------+--------------------------+-----------------+----------------+---------------------+--------------+-----------------+---------------------------+--------------------+--------------+------------+---------------------------+----------------+----------------+-----------------+------------+---------------+------------+---------------------+------------------+-------------------------+--------------------+-------------------+
|provider_id|provider_name|npi|dea|specialty_concept_id|specialty_source_value|care_site_id|year_of_birth|gender_concept_id|care_site_name|place_of_service_concept_id|procedure_count

25/11/23 22:24:42 ERROR ContextFactory: Query execution is null: can't emit event for executionId 412
25/11/23 22:24:42 ERROR ContextFactory: Query execution is null: can't emit event for executionId 412
25/11/23 22:24:42 ERROR ContextFactory: Query execution is null: can't emit event for executionId 412
25/11/23 22:24:42 ERROR ContextFactory: Query execution is null: can't emit event for executionId 413
25/11/23 22:24:42 ERROR ContextFactory: Query execution is null: can't emit event for executionId 413
25/11/23 22:24:42 ERROR ContextFactory: Query execution is null: can't emit event for executionId 413
25/11/23 22:24:42 ERROR ContextFactory: Query execution is null: can't emit event for executionId 414
25/11/23 22:24:42 ERROR ContextFactory: Query execution is null: can't emit event for executionId 414
25/11/23 22:24:43 ERROR ContextFactory: Query execution is null: can't emit event for executionId 415
25/11/23 22:24:43 ERROR ContextFactory: Query execution is null: can't emit event 

In [153]:
print("Columns in gold_comprehensive_scorecard:")
print(gold_comprehensive_scorecard.columns)
print("\nColumn count:", len(gold_comprehensive_scorecard.columns))

Columns in gold_comprehensive_scorecard:
['provider_id', 'provider_name', 'npi', 'dea', 'specialty_concept_id', 'specialty_source_value', 'care_site_id', 'year_of_birth', 'gender_concept_id', 'care_site_name', 'place_of_service_concept_id', 'procedure_count', 'unique_patients_procedures', 'procedure_variety', 'drug_count', 'unique_patients_drugs', 'drug_variety', 'condition_count', 'unique_patients_conditions', 'condition_variety', 'total_encounters', 'total_unique_patients', 'total_patients', 'deceased_patients', 'avg_condition_duration_days', 'condition_complexity', 'mortality_rate', 'provider_age', 'patient_per_encounter_ratio', 'efficiency_score', 'efficiency_grade', 'quality_composite', 'quality_tier', 'influence_score', 'network_role', 'network_density_score', 'collaboration_type', 'overall_performance_score', 'performance_category', 'processed_timestamp']

Column count: 40


In [154]:
print("\n" + "="*60)
print("GOLD 15: Strategic Provider Segmentation")
print("="*60)

# Calculate is_high_volume on the fly using total_encounters
volume_window = W.orderBy(F.col("total_encounters"))
gold_strategic_segmentation = gold_comprehensive_scorecard \
    .withColumn("volume_percentile", F.percent_rank().over(volume_window)) \
    .withColumn("is_high_volume", F.when(F.col("volume_percentile") >= 0.75, 1).otherwise(0)) \
    .withColumn("segment",
                F.when((F.col("efficiency_grade") == "A") & (F.col("quality_tier") == "Tier 1"), "Star Performer")
                 .when((F.col("network_role") == "Hub") & (F.col("collaboration_type") == "High Collaboration"), "Network Leader")
                 .when((F.col("efficiency_score") >= 60) & (F.col("quality_composite") >= 50), "Solid Performer")
                 .when(F.col("is_high_volume") == 1, "High Volume")
                 .otherwise("Emerging Provider")) \
    .withColumn("strategic_priority",
                F.when(F.col("segment") == "Star Performer", "Retain & Reward")
                 .when(F.col("segment") == "Network Leader", "Expand Network")
                 .when(F.col("segment") == "Solid Performer", "Maintain")
                 .when(F.col("segment") == "High Volume", "Quality Improvement")
                 .otherwise("Develop")) \
    .withColumn("processed_timestamp", F.lit(datetime.now().isoformat()))

print(f"✓ Strategic segmentation: {gold_strategic_segmentation.count()} providers")
gold_strategic_segmentation.show(5, truncate=False)

gold_strategic_segmentation = add_layer_metadata(
    df=gold_strategic_segmentation,
    layer="gold",
    source_tables=["gold_comprehensive_scorecard"],
    target_table="gold_strategic_segmentation"
)

# Write immediately!
gold_strategic_segmentation.write.mode("overwrite").parquet("./output/gold/gold_strategic_segmentation")
print(f"✓ SS written")

# Read back
gold_strategic_segmentation = spark.read.parquet("./output/gold/gold_strategic_segmentation")
print(f"  SS: {gold_strategic_segmentation.count()}")

25/11/23 22:24:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/23 22:24:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/23 22:24:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.



GOLD 15: Strategic Provider Segmentation
✓ Strategic segmentation: 0 providers
+-----------+-------------+---+---+--------------------+----------------------+------------+-------------+-----------------+--------------+---------------------------+---------------+--------------------------+-----------------+----------+---------------------+------------+---------------+--------------------------+-----------------+----------------+---------------------+--------------+-----------------+---------------------------+--------------------+--------------+------------+---------------------------+----------------+----------------+-----------------+------------+---------------+------------+---------------------+------------------+-------------------------+--------------------+-------------------+-----------------+--------------+-------+------------------+
|provider_id|provider_name|npi|dea|specialty_concept_id|specialty_source_value|care_site_id|year_of_birth|gender_concept_id|care_site_name|place_

25/11/23 22:24:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/23 22:24:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/23 22:24:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/23 22:24:43 ERROR ContextFactory: Query execution is null: can't emit event for executionId 416
25/11/23 22:24:43 ERROR ContextFactory: Query execution is null: can't emit event for executionId 416
25/11/23 22:24:43 ERROR ContextFactory: Query execution is null: can't emit event for executionId 416
25/11/23 22:24:43 ERROR ContextFactory: Query execution is null: can't emit event for executionId 417
25/11/23 22:24:43 ERROR ContextFactory: Query execution is null: can't emit event for executionId

# STEP 5: Final Metrics (4 Ultimate KPIs)

In [155]:
print("\n" + "="*60)
print("FINAL METRIC 1: Provider Quality Composite")
print("="*60)

final_quality_composite = gold_comprehensive_scorecard \
    .select(
        "provider_id",
        "provider_name",
        "specialty_source_value",
        "quality_composite",
        "quality_tier",
        "efficiency_score",
        "efficiency_grade"
    ) \
    .withColumn("integrated_quality_score",
                (F.col("quality_composite") * 0.6 + F.col("efficiency_score") * 0.4)) \
    .withColumn("quality_status",
                F.when(F.col("integrated_quality_score") >= 75, "High Quality")
                 .when(F.col("integrated_quality_score") >= 50, "Standard Quality")
                 .otherwise("Quality Opportunity")) \
    .orderBy(F.col("integrated_quality_score").desc())

print(f"\n✓ METRIC 1 COMPLETE")
print(f"Providers analyzed: {final_quality_composite.count()}")
print(f"\nTop 5 Quality Performers:")
final_quality_composite.show(5, truncate=False)

final_quality_composite = add_layer_metadata(
    df=final_quality_composite,
    layer="gold",
    source_tables=["gold_comprehensive_scorecard"],
    target_table="final_metric_quality_composite"
)

# Write immediately!
final_quality_composite.write.mode("overwrite").parquet("./output/gold/final_quality_composite")
print(f"✓ QC written")

# Read back
final_quality_composite = spark.read.parquet("./output/gold/final_quality_composite")
print(f"  QC: {final_quality_composite.count()}")

25/11/23 22:24:43 ERROR ContextFactory: Query execution is null: can't emit event for executionId 422
25/11/23 22:24:43 ERROR ContextFactory: Query execution is null: can't emit event for executionId 422
25/11/23 22:24:43 ERROR ContextFactory: Query execution is null: can't emit event for executionId 422
25/11/23 22:24:43 ERROR ContextFactory: Query execution is null: can't emit event for executionId 423
25/11/23 22:24:43 ERROR ContextFactory: Query execution is null: can't emit event for executionId 423
25/11/23 22:24:43 ERROR ContextFactory: Query execution is null: can't emit event for executionId 423
25/11/23 22:24:43 ERROR ContextFactory: Query execution is null: can't emit event for executionId 424
25/11/23 22:24:43 ERROR ContextFactory: Query execution is null: can't emit event for executionId 424



FINAL METRIC 1: Provider Quality Composite

✓ METRIC 1 COMPLETE
Providers analyzed: 0

Top 5 Quality Performers:
+-----------+-------------+----------------------+-----------------+------------+----------------+----------------+------------------------+--------------+
|provider_id|provider_name|specialty_source_value|quality_composite|quality_tier|efficiency_score|efficiency_grade|integrated_quality_score|quality_status|
+-----------+-------------+----------------------+-----------------+------------+----------------+----------------+------------------------+--------------+
+-----------+-------------+----------------------+-----------------+------------+----------------+----------------+------------------------+--------------+

✓ QC written
  QC: 0


In [156]:
print("\n" + "="*60)
print("FINAL METRIC 2: Network Efficiency Score")
print("="*60)

final_network_efficiency = gold_network_efficiency \
    .join(
        gold_network_influence.select(
            F.col("provider_id").alias("prov_id"),
            F.col("network_role").alias("net_role"),
            F.col("referral_direction").alias("ref_dir")
        ), 
        gold_network_efficiency.provider_id == F.col("prov_id"),
        "left"
    ) \
    .select(
        gold_network_efficiency.provider_id,
        F.col("net_role").alias("network_role"),
        F.col("ref_dir").alias("referral_direction"),
        gold_network_efficiency.efficiency_ratio,
        gold_network_efficiency.quality_per_referral,
        gold_network_efficiency.is_efficient_network_provider
    ) \
    .withColumn("network_efficiency_score",
                (F.col("efficiency_ratio") * 50 + F.col("quality_per_referral") * 50)) \
    .withColumn("efficiency_rating",
                F.when(F.col("network_efficiency_score") >= 75, "Highly Efficient")
                 .when(F.col("network_efficiency_score") >= 50, "Efficient")
                 .when(F.col("network_efficiency_score") >= 25, "Moderately Efficient")
                 .otherwise("Inefficient")) \
    .orderBy(F.col("network_efficiency_score").desc())

print(f"\n✓ METRIC 2 COMPLETE")
print(f"Providers in network: {final_network_efficiency.count()}")
print(f"\nTop 5 Network Efficient Providers:")
final_network_efficiency.show(5, truncate=False)

final_network_efficiency = add_layer_metadata(
    df=final_network_efficiency,
    layer="gold",
    source_tables=["gold_network_efficiency", "gold_network_influence"],
    target_table="final_metric_network_efficiency"
)

# Write immediately!
final_network_efficiency.write.mode("overwrite").parquet("./output/gold/final_network_efficiency")
print(f"✓ NE written")

# Read back
final_network_efficiency = spark.read.parquet("./output/gold/final_network_efficiency")
print(f"  NE: {final_network_efficiency.count()}")


FINAL METRIC 2: Network Efficiency Score

✓ METRIC 2 COMPLETE
Providers in network: 2

Top 5 Network Efficient Providers:
+-----------+------------+------------------+----------------+--------------------+-----------------------------+------------------------+-----------------+
|provider_id|network_role|referral_direction|efficiency_ratio|quality_per_referral|is_efficient_network_provider|network_efficiency_score|efficiency_rating|
+-----------+------------+------------------+----------------+--------------------+-----------------------------+------------------------+-----------------+
|2632       |Connector   |Outbound          |50.0            |50.0                |1                            |5000.0                  |Highly Efficient |
|61665      |Connector   |Outbound          |46.0            |46.0                |1                            |4600.0                  |Highly Efficient |
+-----------+------------+------------------+----------------+--------------------+---------

25/11/23 22:24:43 ERROR ContextFactory: Query execution is null: can't emit event for executionId 426
25/11/23 22:24:43 ERROR ContextFactory: Query execution is null: can't emit event for executionId 426
25/11/23 22:24:43 ERROR ContextFactory: Query execution is null: can't emit event for executionId 426
25/11/23 22:24:43 ERROR ContextFactory: Query execution is null: can't emit event for executionId 427
25/11/23 22:24:43 ERROR ContextFactory: Query execution is null: can't emit event for executionId 427
25/11/23 22:24:43 ERROR ContextFactory: Query execution is null: can't emit event for executionId 427
25/11/23 22:24:43 ERROR ContextFactory: Query execution is null: can't emit event for executionId 427
25/11/23 22:24:43 ERROR ContextFactory: Query execution is null: can't emit event for executionId 428
25/11/23 22:24:43 ERROR ContextFactory: Query execution is null: can't emit event for executionId 428
25/11/23 22:24:43 ERROR ContextFactory: Query execution is null: can't emit event 

In [157]:
print("\n" + "="*60)
print("FINAL METRIC 3: Patient Outcome Attribution")
print("="*60)

final_outcome_attribution = gold_outcome_attribution \
    .select(
        "provider_id",
        "outcome_tier",
        "outcome_contribution_score",
        "complexity_adjusted_attribution",
        "attribution_weight",
        "total_patients",
        "condition_complexity"
    ) \
    .withColumn("patient_outcome_impact",
                F.col("outcome_contribution_score") * F.col("complexity_adjusted_attribution")) \
    .withColumn("attribution_category",
                F.when(F.col("patient_outcome_impact") >= 1000, "High Impact")
                 .when(F.col("patient_outcome_impact") >= 500, "Moderate Impact")
                 .when(F.col("patient_outcome_impact") >= 100, "Standard Impact")
                 .otherwise("Emerging Impact")) \
    .orderBy(F.col("patient_outcome_impact").desc())

print(f"\n✓ METRIC 3 COMPLETE")
print(f"Providers with outcome data: {final_outcome_attribution.count()}")
print(f"\nTop 5 Outcome Contributors:")
final_outcome_attribution.show(5, truncate=False)

final_outcome_attribution = add_layer_metadata(
    df=final_outcome_attribution,
    layer="gold",
    source_tables=["gold_outcome_attribution"],
    target_table="final_metric_outcome_attribution"
)

# Write immediately!
final_outcome_attribution.write.mode("overwrite").parquet("./output/gold/final_outcome_attribution")
print(f"✓ OA written")

# Read back
final_outcome_attribution = spark.read.parquet("./output/gold/final_outcome_attribution")
print(f"  OA: {final_outcome_attribution.count()}")

25/11/23 22:24:43 ERROR ContextFactory: Query execution is null: can't emit event for executionId 430
25/11/23 22:24:43 ERROR ContextFactory: Query execution is null: can't emit event for executionId 430
25/11/23 22:24:43 ERROR ContextFactory: Query execution is null: can't emit event for executionId 430



FINAL METRIC 3: Patient Outcome Attribution

✓ METRIC 3 COMPLETE
Providers with outcome data: 1

Top 5 Outcome Contributors:
+-----------+------------+--------------------------+-------------------------------+------------------+--------------+--------------------+----------------------+--------------------+
|provider_id|outcome_tier|outcome_contribution_score|complexity_adjusted_attribution|attribution_weight|total_patients|condition_complexity|patient_outcome_impact|attribution_category|
+-----------+------------+--------------------------+-------------------------------+------------------+--------------+--------------------+----------------------+--------------------+
|9069       |Excellent   |25.0                      |12.5                           |0.25              |1             |1                   |312.5                 |Standard Impact     |
+-----------+------------+--------------------------+-------------------------------+------------------+--------------+---------------

In [158]:
print("\n" + "="*60)
print("FINAL METRIC 4: Care Coordination Index")
print("="*60)

final_care_coordination = gold_collaboration_density \
    .select(
        "provider_id",
        "collaboration_type",
        "network_density_score",
        "avg_providers_per_patient",
        "patients_requiring_coordination",
        "team_based_care_indicator",
        "collaboration_intensity"
    ) \
    .withColumn("coordination_index",
                (F.col("network_density_score") * 0.4 +
                 F.col("collaboration_intensity") * 0.3 +
                 F.col("avg_providers_per_patient") * 10 * 0.3)) \
    .withColumn("coordination_excellence",
                F.when(F.col("coordination_index") >= 50, "Excellent Coordination")
                 .when(F.col("coordination_index") >= 30, "Good Coordination")
                 .when(F.col("coordination_index") >= 15, "Adequate Coordination")
                 .otherwise("Limited Coordination")) \
    .orderBy(F.col("coordination_index").desc())

print(f"\n✓ METRIC 4 COMPLETE")
print(f"Providers with coordination data: {final_care_coordination.count()}")
print(f"\nTop 5 Care Coordinators:")
final_care_coordination.show(5, truncate=False)

final_care_coordination = add_layer_metadata(
    df=final_care_coordination,
    layer="gold",
    source_tables=["gold_collaboration_density"],
    target_table="final_metric_care_coordination"
)

# Write immediately!
final_care_coordination.write.mode("overwrite").parquet("./output/gold/final_care_coordination")
print(f"✓ CC written")

# Read back
final_care_coordination = spark.read.parquet("./output/gold/final_care_coordination")
print(f"  CC: {final_care_coordination.count()}")


FINAL METRIC 4: Care Coordination Index

✓ METRIC 4 COMPLETE
Providers with coordination data: 798

Top 5 Care Coordinators:
+-----------+----------------------+---------------------+-------------------------+-------------------------------+-------------------------+-----------------------+------------------+-----------------------+
|provider_id|collaboration_type    |network_density_score|avg_providers_per_patient|patients_requiring_coordination|team_based_care_indicator|collaboration_intensity|coordination_index|coordination_excellence|
+-----------+----------------------+---------------------+-------------------------+-------------------------------+-------------------------+-----------------------+------------------+-----------------------+
|250784     |Moderate Collaboration|0.04                 |2.0                      |1                              |0                        |2.0                    |6.616             |Limited Coordination   |
|26266      |Moderate Collaboratio

25/11/23 22:24:44 ERROR ContextFactory: Query execution is null: can't emit event for executionId 434
25/11/23 22:24:44 ERROR ContextFactory: Query execution is null: can't emit event for executionId 434
25/11/23 22:24:44 ERROR ContextFactory: Query execution is null: can't emit event for executionId 434
25/11/23 22:24:44 ERROR ContextFactory: Query execution is null: can't emit event for executionId 435
25/11/23 22:24:44 ERROR ContextFactory: Query execution is null: can't emit event for executionId 435
25/11/23 22:24:44 ERROR ContextFactory: Query execution is null: can't emit event for executionId 435
25/11/23 22:24:44 ERROR ContextFactory: Query execution is null: can't emit event for executionId 436
25/11/23 22:24:44 ERROR ContextFactory: Query execution is null: can't emit event for executionId 436
25/11/23 22:24:44 ERROR ContextFactory: Query execution is null: can't emit event for executionId 437
25/11/23 22:24:44 ERROR ContextFactory: Query execution is null: can't emit event 

# STEP 6: Build DAG (Bronze → Silver 6 → Gold 15 → Metrics 4)

In [159]:
print("\n" + "="*60)
print("BUILDING DAG")
print("="*60)

G = nx.DiGraph()

bronze_nodes = [
    "bronze_provider", "bronze_care_site", "bronze_person", "bronze_death",
    "bronze_procedure", "bronze_drug", "bronze_condition", "bronze_observation",
    "bronze_medicare", "bronze_taxonomy"
]

silver_nodes = [
    "silver_provider_demographics", "silver_provider_activity", 
    "silver_patient_outcomes", "silver_referral_network",
    "silver_quality_indicators", "silver_care_coordination"
]

gold_nodes = [
    "gold_provider_master", "gold_volume_metrics", "gold_clinical_diversity",
    "gold_outcome_tiers", "gold_network_connectivity", "gold_quality_subscores",
    "gold_coordination_effectiveness", "gold_efficiency_index", "gold_network_influence",
    "gold_quality_adjusted", "gold_collaboration_density", "gold_outcome_attribution",
    "gold_network_efficiency", "gold_comprehensive_scorecard", "gold_strategic_segmentation"
]

metric_nodes = [
    "final_quality_composite", "final_network_efficiency",
    "final_outcome_attribution", "final_care_coordination"
]

for node in bronze_nodes:
    G.add_node(node, layer="bronze", label=node)
    
for node in silver_nodes:
    G.add_node(node, layer="silver", label=node)
    
for node in gold_nodes:
    G.add_node(node, layer="gold", label=node)
    
for node in metric_nodes:
    G.add_node(node, layer="metric", label=node)

G.add_edge("bronze_provider", "silver_provider_demographics")
G.add_edge("bronze_care_site", "silver_provider_demographics")
G.add_edge("bronze_taxonomy", "silver_provider_demographics")

G.add_edge("bronze_procedure", "silver_provider_activity")
G.add_edge("bronze_drug", "silver_provider_activity")
G.add_edge("bronze_condition", "silver_provider_activity")

G.add_edge("bronze_person", "silver_patient_outcomes")
G.add_edge("bronze_death", "silver_patient_outcomes")
G.add_edge("bronze_condition", "silver_patient_outcomes")

G.add_edge("bronze_procedure", "silver_referral_network")
G.add_edge("bronze_drug", "silver_referral_network")

G.add_edge("bronze_condition", "silver_quality_indicators")

G.add_edge("bronze_condition", "silver_care_coordination")
G.add_edge("bronze_procedure", "silver_care_coordination")
G.add_edge("bronze_drug", "silver_care_coordination")

G.add_edge("silver_provider_demographics", "gold_provider_master")
G.add_edge("silver_provider_activity", "gold_provider_master")
G.add_edge("silver_patient_outcomes", "gold_provider_master")

G.add_edge("silver_provider_activity", "gold_volume_metrics")
G.add_edge("silver_provider_activity", "gold_clinical_diversity")

G.add_edge("silver_patient_outcomes", "gold_outcome_tiers")

G.add_edge("silver_referral_network", "gold_network_connectivity")

G.add_edge("silver_quality_indicators", "gold_quality_subscores")

G.add_edge("silver_care_coordination", "gold_coordination_effectiveness")

G.add_edge("gold_provider_master", "gold_efficiency_index")
G.add_edge("gold_volume_metrics", "gold_efficiency_index")
G.add_edge("gold_outcome_tiers", "gold_efficiency_index")

G.add_edge("gold_network_connectivity", "gold_network_influence")

G.add_edge("gold_quality_subscores", "gold_quality_adjusted")

G.add_edge("gold_coordination_effectiveness", "gold_collaboration_density")

G.add_edge("gold_outcome_tiers", "gold_outcome_attribution")
G.add_edge("gold_volume_metrics", "gold_outcome_attribution")

G.add_edge("gold_network_influence", "gold_network_efficiency")
G.add_edge("gold_quality_adjusted", "gold_network_efficiency")

G.add_edge("gold_provider_master", "gold_comprehensive_scorecard")
G.add_edge("gold_efficiency_index", "gold_comprehensive_scorecard")
G.add_edge("gold_quality_adjusted", "gold_comprehensive_scorecard")
G.add_edge("gold_network_influence", "gold_comprehensive_scorecard")
G.add_edge("gold_collaboration_density", "gold_comprehensive_scorecard")

G.add_edge("gold_comprehensive_scorecard", "gold_strategic_segmentation")

G.add_edge("gold_comprehensive_scorecard", "final_quality_composite")

G.add_edge("gold_network_efficiency", "final_network_efficiency")
G.add_edge("gold_network_influence", "final_network_efficiency")

G.add_edge("gold_outcome_attribution", "final_outcome_attribution")
G.add_edge("gold_outcome_tiers", "final_outcome_attribution")

G.add_edge("gold_collaboration_density", "final_care_coordination")
G.add_edge("gold_coordination_effectiveness", "final_care_coordination")

print(f"✓ DAG constructed:")
print(f"  Nodes: {G.number_of_nodes()}")
print(f"  Edges: {G.number_of_edges()}")
print(f"  Layers: Bronze(10) → Silver(6) → Gold(15) → Metrics(4)")
print(f"  Is DAG: {nx.is_directed_acyclic_graph(G)}")


BUILDING DAG
✓ DAG constructed:
  Nodes: 35
  Edges: 47
  Layers: Bronze(10) → Silver(6) → Gold(15) → Metrics(4)
  Is DAG: True


In [160]:
print("\n" + "="*60)
print("GENERATE RAG DATA")
print("="*60)

rag_data = []

for node_id in G.nodes():
    node_attrs = G.nodes[node_id]
    label = node_attrs.get('label', '')
    layer = node_attrs.get('layer', 'unknown')
    
    in_degree = G.in_degree(node_id)
    out_degree = G.out_degree(node_id)
    
    parents = list(G.predecessors(node_id))
    children = list(G.successors(node_id))
    
    texts = [
        f"{label}",
        f"Layer: {layer}",
        f"Incoming: {in_degree}, Outgoing: {out_degree}"
    ]
    
    if parents:
        texts.append(f"Consumes: {', '.join(parents[:5])}")
    
    if children:
        texts.append(f"Feeds into: {', '.join(children[:5])}")
    
    rag_data.append({
        "id": node_id,
        "texts": texts
    })

print(f"✓ Generated RAG data for {len(rag_data)} nodes")


GENERATE RAG DATA
✓ Generated RAG data for 35 nodes


In [161]:
print("\n" + "="*60)
print("SAVE DAG AND RAG DATA")
print("="*60)

dag_file = f"{LOCAL_DATA_DIR}/provider_performance_dag.graphml"
nx.write_graphml(G, dag_file)
print(f"✓ Saved DAG: {dag_file}")

rag_file = f"{LOCAL_DATA_DIR}/provider_performance_rag_data.json"
with open(rag_file, 'w') as f:
    json.dump(rag_data, f, indent=2)
print(f"✓ Saved RAG data: {rag_file}")

stats = {
    "total_nodes": G.number_of_nodes(),
    "total_edges": G.number_of_edges(),
    "bronze_nodes": len(bronze_nodes),
    "silver_nodes": len(silver_nodes),
    "gold_nodes": len(gold_nodes),
    "metric_nodes": len(metric_nodes),
    "is_dag": nx.is_directed_acyclic_graph(G),
    "timestamp": datetime.now().isoformat()
}

stats_file = f"{LOCAL_DATA_DIR}/dag_statistics.json"
with open(stats_file, 'w') as f:
    json.dump(stats, f, indent=2)
print(f"✓ Saved statistics: {stats_file}")


SAVE DAG AND RAG DATA
✓ Saved DAG: ./4_data/provider_performance_dag.graphml
✓ Saved RAG data: ./4_data/provider_performance_rag_data.json
✓ Saved statistics: ./4_data/dag_statistics.json


25/11/23 22:24:44 ERROR ContextFactory: Query execution is null: can't emit event for executionId 438
25/11/23 22:24:44 ERROR ContextFactory: Query execution is null: can't emit event for executionId 438
25/11/23 22:24:44 ERROR ContextFactory: Query execution is null: can't emit event for executionId 438


# STEP 7: Summary

In [162]:
print("\n" + "="*80)
print("PROVIDER PERFORMANCE & NETWORK ANALYTICS COMPLETE")
print("="*80)

print(f"\nData Sources:")
print(f"  OMOP Clinical (8 tables)")
print(f"  Medicare Provider (1 table)")
print(f"  NPPES Taxonomy (1 table)")

print(f"\nPipeline Architecture:")
print(f"  Bronze Layer: {len(bronze_nodes)} tables (raw data)")
print(f"  Silver Layer: {len(silver_nodes)} tables (provider profiling)")
print(f"  Gold Layer: {len(gold_nodes)} tables (vertical analytics)")
print(f"  Final Metrics: {len(metric_nodes)} KPIs")

print(f"\nFinal Metrics:")
print(f"  1. Provider Quality Composite: {final_quality_composite.count()} providers")
print(f"  2. Network Efficiency Score: {final_network_efficiency.count()} providers")
print(f"  3. Patient Outcome Attribution: {final_outcome_attribution.count()} providers")
print(f"  4. Care Coordination Index: {final_care_coordination.count()} providers")

print(f"\nDAG:")
print(f"  Total nodes: {G.number_of_nodes()}")
print(f"  Total edges: {G.number_of_edges()}")
print(f"  Format: GraphML + RAG JSON")

print(f"\nCost Optimization:")
print(f"  Strategy: LIMIT {LIMIT} on all BigQuery reads")
print(f"  Storage: Local CSV (Pandas → Spark)")
print(f"  Total tables processed: {len(bronze_nodes)}")

print("\n" + "="*80)
print("✓ Ready for Marquez lineage tracking")
print("✓ Ready for RAG pipeline integration")
print("="*80)


PROVIDER PERFORMANCE & NETWORK ANALYTICS COMPLETE

Data Sources:
  OMOP Clinical (8 tables)
  Medicare Provider (1 table)
  NPPES Taxonomy (1 table)

Pipeline Architecture:
  Bronze Layer: 10 tables (raw data)
  Silver Layer: 6 tables (provider profiling)
  Gold Layer: 15 tables (vertical analytics)
  Final Metrics: 4 KPIs

Final Metrics:
  1. Provider Quality Composite: 0 providers
  2. Network Efficiency Score: 2 providers
  3. Patient Outcome Attribution: 1 providers
  4. Care Coordination Index: 798 providers

DAG:
  Total nodes: 35
  Total edges: 47
  Format: GraphML + RAG JSON

Cost Optimization:
  Strategy: LIMIT 1000 on all BigQuery reads
  Storage: Local CSV (Pandas → Spark)
  Total tables processed: 10

✓ Ready for Marquez lineage tracking
✓ Ready for RAG pipeline integration


In [163]:
# Verify Lineage
print("\n" + "="*60)
print("LINEAGE TRACKING COMPLETE!")
print("="*60)
print(f"✅ Notebook completed successfully")
print(f"✅ Marquez Web UI: http://localhost:3601")
print(f"✅ Namespace: provider_performance")  # CORRECTED!
print(f"\nNext Steps:")
print("1. Open http://localhost:3601 in your browser")
print("2. Select namespace: 'provider_performance'")
print("3. Browse Jobs and Datasets")
print("4. Click on any dataset to see lineage graph")
print("="*60)

# CRITICAL: Stop Spark to send job completion event
print("\nStopping Spark session to complete job...")
spark.stop()
print("✅ Spark session stopped - job marked as COMPLETE in Marquez")


LINEAGE TRACKING COMPLETE!
✅ Notebook completed successfully
✅ Marquez Web UI: http://localhost:3601
✅ Namespace: provider_performance

Next Steps:
1. Open http://localhost:3601 in your browser
2. Select namespace: 'provider_performance'
3. Browse Jobs and Datasets
4. Click on any dataset to see lineage graph

Stopping Spark session to complete job...
✅ Spark session stopped - job marked as COMPLETE in Marquez
